# MFD: Polarization-Resolved Filtered FCS

This notebook demonstrates MFD (Multi-parameter Fluorescence Detection) mode for computing lifetime filters from polarization-resolved data.

## When to Use MFD Mode

MFD mode is essential when:
- You have parallel and perpendicular polarization channels
- Species have different anisotropies (rotational diffusion)
- You need anisotropy-resolved dynamics
- You're measuring FRET with polarization

## MFD vs Single-Channel

- **Single-channel**: Combines all photons regardless of polarization
- **MFD**: Treats parallel and perpendicular channels independently
- **Advantage**: Better species separation when anisotropies differ
- **Requirement**: Need decay patterns for both polarizations

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Import MFD API
import sys
sys.path.insert(0, str(Path.cwd().parent.parent.parent.parent))
from chisurf.plugins.fcs.fcs_filter_calculator import (
    compute_filters_mfd,
    FilterResultMFD,
    DetectionMode
)

## Generate Synthetic MFD Data

We'll create two species with:
- Different lifetimes (τ₁ = 1 ns, τ₂ = 3 ns)
- Different anisotropies (r₁ = 0.1, r₂ = 0.3)

The parallel and perpendicular intensities follow:
- I_∥ = I₀ (1 + 2r)
- I_⊥ = I₀ (1 - r)

In [ ]:
# Parameters
n_bins = 256
time = np.linspace(0, 20, n_bins)

# Lifetimes
tau1, tau2 = 1.0, 3.0  # nanoseconds

# Anisotropies
r1, r2 = 0.1, 0.3

# Isotropic decays
I1_iso = np.exp(-time / tau1)
I2_iso = np.exp(-time / tau2)

# Parallel channel: I_par = I_iso * (1 + 2r)
I1_par = I1_iso * (1 + 2*r1)
I2_par = I2_iso * (1 + 2*r2)

# Perpendicular channel: I_perp = I_iso * (1 - r)
I1_perp = I1_iso * (1 - r1)
I2_perp = I2_iso * (1 - r2)

# Normalize and scale
I1_par = I1_par / I1_par.sum() * 10000
I2_par = I2_par / I2_par.sum() * 10000
I1_perp = I1_perp / I1_perp.sum() * 10000
I2_perp = I2_perp / I2_perp.sum() * 10000

# Mixture (60% species 1, 40% species 2)
w1, w2 = 0.6, 0.4
total_par = w1 * I1_par + w2 * I2_par
total_perp = w1 * I1_perp + w2 * I2_perp

# Add Poisson noise
rng = np.random.RandomState(42)
total_par_noisy = rng.poisson(total_par)
total_perp_noisy = rng.poisson(total_perp)
I1_par_noisy = rng.poisson(I1_par)
I2_par_noisy = rng.poisson(I2_par)
I1_perp_noisy = rng.poisson(I1_perp)
I2_perp_noisy = rng.poisson(I2_perp)

print(f"Generated MFD data:")
print(f"  Species 1: τ={tau1} ns, r={r1}")
print(f"  Species 2: τ={tau2} ns, r={r2}")
print(f"  Mixture: {w1*100:.0f}% sp1 + {w2*100:.0f}% sp2")
print(f"  Photons (par): {total_par_noisy.sum():.0f}")
print(f"  Photons (perp): {total_perp_noisy.sum():.0f}")

## Visualize Polarization-Dependent Decays

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Parallel channel
ax = axes[0, 0]
ax.plot(time, total_par_noisy, 'k-', label='Total (par)', lw=2)
ax.plot(time, I1_par_noisy, 'b--', label=f'Sp1 (r={r1})', alpha=0.7)
ax.plot(time, I2_par_noisy, 'r--', label=f'Sp2 (r={r2})', alpha=0.7)
ax.set_xlabel('Time (ns)')
ax.set_ylabel('Counts')
ax.set_title('Parallel Channel (Linear)')
ax.legend()
ax.grid(alpha=0.3)

ax = axes[0, 1]
ax.semilogy(time, total_par_noisy, 'k-', label='Total (par)', lw=2)
ax.semilogy(time, I1_par_noisy, 'b--', label=f'Sp1', alpha=0.7)
ax.semilogy(time, I2_par_noisy, 'r--', label=f'Sp2', alpha=0.7)
ax.set_xlabel('Time (ns)')
ax.set_ylabel('Counts (log)')
ax.set_title('Parallel Channel (Log)')
ax.legend()
ax.grid(alpha=0.3)

# Perpendicular channel
ax = axes[1, 0]
ax.plot(time, total_perp_noisy, 'k-', label='Total (perp)', lw=2)
ax.plot(time, I1_perp_noisy, 'b--', label=f'Sp1 (r={r1})', alpha=0.7)
ax.plot(time, I2_perp_noisy, 'r--', label=f'Sp2 (r={r2})', alpha=0.7)
ax.set_xlabel('Time (ns)')
ax.set_ylabel('Counts')
ax.set_title('Perpendicular Channel (Linear)')
ax.legend()
ax.grid(alpha=0.3)

ax = axes[1, 1]
ax.semilogy(time, total_perp_noisy, 'k-', label='Total (perp)', lw=2)
ax.semilogy(time, I1_perp_noisy, 'b--', label=f'Sp1', alpha=0.7)
ax.semilogy(time, I2_perp_noisy, 'r--', label=f'Sp2', alpha=0.7)
ax.set_xlabel('Time (ns)')
ax.set_ylabel('Counts (log)')
ax.set_title('Perpendicular Channel (Log)')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Compute MFD Filters

In [ ]:
# Compute MFD filters
result = compute_filters_mfd(
    total_decay_par=total_par_noisy,
    total_decay_perp=total_perp_noisy,
    species_decays_par=[I1_par_noisy, I2_par_noisy],
    species_decays_perp=[I1_perp_noisy, I2_perp_noisy],
    metadata={
        "tau1_ns": tau1,
        "tau2_ns": tau2,
        "r1": r1,
        "r2": r2,
        "w1": w1,
        "w2": w2,
    }
)

print(f"MFD Filter Computation:")
print(f"  Mode: {result.mode.value}")
print(f"  Species: {result.n_species}")
print(f"  TAC bins: {result.n_bins}")
print(f"  Filters (par): {result.filters_par.shape}")
print(f"  Filters (perp): {result.filters_perp.shape}")
print(f"  Max residual (par): {np.abs(result.weighted_residuals_par).max():.3f}")
print(f"  Max residual (perp): {np.abs(result.weighted_residuals_perp).max():.3f}")

## Visualize MFD Filters

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Parallel filters
ax = axes[0, 0]
ax.plot(time, result.filters_par[0, :], 'b-', label='Filter 1 (short τ, low r)', lw=2)
ax.plot(time, result.filters_par[1, :], 'r-', label='Filter 2 (long τ, high r)', lw=2)
ax.axhline(0, color='k', linestyle='--', alpha=0.3)
ax.set_xlabel('Time (ns)')
ax.set_ylabel('Filter value')
ax.set_title('Parallel Channel Filters')
ax.legend()
ax.grid(alpha=0.3)

# Perpendicular filters
ax = axes[0, 1]
ax.plot(time, result.filters_perp[0, :], 'b-', label='Filter 1', lw=2)
ax.plot(time, result.filters_perp[1, :], 'r-', label='Filter 2', lw=2)
ax.axhline(0, color='k', linestyle='--', alpha=0.3)
ax.set_xlabel('Time (ns)')
ax.set_ylabel('Filter value')
ax.set_title('Perpendicular Channel Filters')
ax.legend()
ax.grid(alpha=0.3)

# Parallel reconstruction
ax = axes[1, 0]
ax.semilogy(time, total_par_noisy, 'ko', markersize=3, alpha=0.5, label='Data')
ax.semilogy(time, result.reconstruction_par, 'r-', lw=2, label='Reconstruction')
ax.set_xlabel('Time (ns)')
ax.set_ylabel('Counts (log)')
ax.set_title('Parallel: Reconstruction Quality')
ax.legend()
ax.grid(alpha=0.3)

# Perpendicular reconstruction
ax = axes[1, 1]
ax.semilogy(time, total_perp_noisy, 'ko', markersize=3, alpha=0.5, label='Data')
ax.semilogy(time, result.reconstruction_perp, 'r-', lw=2, label='Reconstruction')
ax.set_xlabel('Time (ns)')
ax.set_ylabel('Counts (log)')
ax.set_title('Perpendicular: Reconstruction Quality')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Compare Filter Differences Between Channels

The filters differ between parallel and perpendicular because species have different anisotropies.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Species 1 filters
ax = axes[0]
ax.plot(time, result.filters_par[0, :], 'b-', label='Parallel', lw=2)
ax.plot(time, result.filters_perp[0, :], 'r-', label='Perpendicular', lw=2)
ax.axhline(0, color='k', linestyle='--', alpha=0.3)
ax.set_xlabel('Time (ns)')
ax.set_ylabel('Filter value')
ax.set_title(f'Species 1 Filters (τ={tau1} ns, r={r1})')
ax.legend()
ax.grid(alpha=0.3)

# Species 2 filters
ax = axes[1]
ax.plot(time, result.filters_par[1, :], 'b-', label='Parallel', lw=2)
ax.plot(time, result.filters_perp[1, :], 'r-', label='Perpendicular', lw=2)
ax.axhline(0, color='k', linestyle='--', alpha=0.3)
ax.set_xlabel('Time (ns)')
ax.set_ylabel('Filter value')
ax.set_title(f'Species 2 Filters (τ={tau2} ns, r={r2})')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Quantify difference
diff1 = np.abs(result.filters_par[0] - result.filters_perp[0]).mean()
diff2 = np.abs(result.filters_par[1] - result.filters_perp[1]).mean()
print(f"Average filter difference between channels:")
print(f"  Species 1: {diff1:.4f}")
print(f"  Species 2: {diff2:.4f}")
print(f"  → Species 2 shows larger difference (higher anisotropy)")

## Export MFD Filters

In [ ]:
# Export to JSON
output_path = "fcs_filter_mfd.json"
result.to_json(output_path, indent=2)

print(f"✓ MFD filters exported to: {output_path}")

# Verify by loading back
loaded = FilterResultMFD.from_json(output_path)
print(f"\n✓ Verified: loaded {loaded.n_species} species, {loaded.n_bins} bins")
print(f"  Mode: {loaded.mode.value}")

## Applying MFD Filters to Photon Data

In practice, you would apply these filters to your TTTR data:

```python
import tttrlib

# Load TTTR file
tttr = tttrlib.TTTR("experiment.ptu", "PTU")
macro = tttr.get_macro_time()
micro = tttr.get_micro_time()
channel = tttr.get_channel()  # 0=parallel, 1=perpendicular

# Apply filters based on polarization
mask_par = (channel == 0)
mask_perp = (channel == 1)

# Species 1 weights
weights1_par = result.filters_par[0, micro[mask_par]]
weights1_perp = result.filters_perp[0, micro[mask_perp]]
weights1 = np.concatenate([weights1_par, weights1_perp])

# Species 2 weights
weights2_par = result.filters_par[1, micro[mask_par]]
weights2_perp = result.filters_perp[1, micro[mask_perp]]
weights2 = np.concatenate([weights2_par, weights2_perp])

# Correlate
correlator = tttrlib.Correlator()
corr1 = correlator.run(macro, weights1)
corr2 = correlator.run(macro, weights2)
```

## Summary

MFD mode provides:
- ✓ **Separate filters** for parallel and perpendicular channels
- ✓ **Better species separation** when anisotropies differ
- ✓ **Anisotropy information** preserved in filters
- ✓ **Compatible with standard fFCS workflow**

## When MFD is Essential

1. **Rotational diffusion** - Species with different rotation rates
2. **FRET dynamics** - Donor/acceptor have different anisotropies
3. **Protein binding** - Free vs bound states differ in rotation
4. **Membrane studies** - Probe orientation matters

## Next Steps

- Apply to real polarization-resolved TTTR data
- Compare MFD vs single-channel filtering
- Analyze anisotropy decay from filter differences